In [1]:
import pandas as pd
import requests
import asyncio
from timeit import default_timer
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
import aiohttp
import asyncio
import pandas as pd
from io import StringIO

currency = "GBP"

# suddividiamo il periodo in blocchi annuali
periods = [
    ("2025-01-01", "2025-03-31"),
    ("2025-04-01", "2025-06-30"),
    ("2025-07-01", "2022-09-30"),
    ("2025-10-01", "2025-12-31"),
    ("2026-01-01", "2026-03-10")
]


In [3]:
def download_fx(start, end):
    url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.{currency}.EUR.SP00.A"
    params = {"startPeriod": start, "endPeriod": end, "format": "csvdata"}
    r = requests.get(url, params=params)
    df = pd.read_csv(StringIO(r.text))
    return df

results = []

# ThreadPoolExecutor per scaricare tutti i blocchi in parallelo
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(download_fx, start, end) for start, end in periods]
    for f in as_completed(futures):
        results.append(f.result())

# concatena tutti i dataframe
df = pd.concat(results).sort_values("TIME_PERIOD").reset_index(drop=True)

# print(df.head())
print("Totale righe:", len(df))

Totale righe: 237


In [4]:
df.head(5).style

In [ ]:
# import requests
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # currencies = ["USD", "GBP", "JPY", "CHF", "AUD", "CAD"]

# start = "2025-01-01"
# end = "2026-03-10"

# def download_fx():
#     url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.GBP.EUR.SP00.A"
#     params = {
#         "startPeriod": start,
#         "endPeriod": end,
#         "format": "csvdata"
#     }

#     r = requests.get(url, params=params)
#     df = pd.read_csv(pd.compat.StringIO(r.text))
#     # df["CURRENCY"] = currency

#     return df

# results = []

# with ThreadPoolExecutor(max_workers=6) as executor:
#     futures = [executor.submit(download_fx, c) for c in currencies]

#     for f in as_completed(futures):
#         results.append(f.result())

# df = pd.concat(results)

# print(df.head())

#Async 

In [13]:
async def fetch(session, start, end):
    url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.{currency}.EUR.SP00.A"
    params = {"startPeriod": start, 
              "endPeriod": end, 
              "format": "csvdata"}
    async with session.get(url, params=params) as response:
        text = await response.text()
        return pd.read_csv(StringIO(text))

In [14]:
async def main():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, start, end) for start, end in periods]
        results = await asyncio.gather(*tasks)
    # concatena tutti i dataframe
    df = pd.concat(results).sort_values("TIME_PERIOD").reset_index(drop=True)
    print(df.head())
    print("Totale righe:", len(df))

In [15]:
asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
import aiohttp
import asyncio
import pandas as pd
from io import StringIO

currency = "GBP"

# suddividiamo il periodo in blocchi annuali
periods = [
    ("2025-01-01", "2025-03-31"),
    ("2025-04-01", "2025-06-30"),
    ("2025-07-01", "2022-09-30"),
    ("2025-10-01", "2025-12-31"),
    ("2026-01-01", "2026-03-10")
]

async def fetch(session, start, end):
    url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.{currency}.EUR.SP00.A"
    params = {"startPeriod": start, 
              "endPeriod": end, 
              "format": "csvdata"}
    async with session.get(url, params=params) as response:
        text = await response.text()
        return pd.read_csv(StringIO(text))

async def main():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, start, end) for start, end in periods]
        results = await asyncio.gather(*tasks)
    # concatena tutti i dataframe
    df = pd.concat(results).sort_values("TIME_PERIOD").reset_index(drop=True)
    print(df.head())
    print("Totale righe:", len(df))

# esegui
asyncio.run(main())

# Back up

In [ ]:
START_TIME = default_timer()

def request(session, i):
    startPeriod = '2025-01-01'
    endPeriod =  '2026-03-13'

    url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.GBP.EUR.SP00.A?startPeriod={startPeriod}&endPeriod={endPeriod}"
    with session.get(url) as response:
        data = response.text

        if response.status_code != 200:
            print("FAILURE::{0}".format(url))

        elapsed_time = default_timer() - START_TIME
        completed_at = "{:5.2f}s".format(elapsed_time)
        print("{0:<30} {1:>20}".format(i, completed_at))
        return data

async def start_async_process():
    print("{0:<30} {1:>20}".format("No", "Completed at"))
    with ThreadPoolExecutor(max_workers=6) as executor:
        with requests.Session() as session:
            loop = asyncio.get_event_loop()
            START_TIME = default_timer()
            tasks = [
                loop.run_in_executor(
                    executor,
                    request,
                    *(session,i)
                )
                for i in range(15)
            ]
            for response in await asyncio.gather(*tasks):
                pass




In [ ]:
loop = asyncio.get_event_loop()

In [ ]:
future = asyncio.ensure_future(start_async_process())


In [ ]:
loop.run_until_complete(future)

In [ ]:
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    future = asyncio.ensure_future(start_async_process())
    loop.run_until_complete(future)